# Algorithms 5.5 / 5.6 — Gauss's method (angles-only orbit determination)

**Goal:** the hardest and most complete OD in the book. From **three lines of sight** to an object (directions only — no ranges) taken at three times by a known, moving observer, recover the object's state vector $(\mathbf{r}_2,\mathbf{v}_2)$ at the middle time.

**Code (answer key):** [`gauss`](../curtis/algorithms/alg_5_5_gauss.py) · **Book:** §5.10, Algorithms 5.5 (initial estimate) + 5.6 (iterative improvement) · **Worked example:** 5.11

This pulls together almost everything: the line-of-sight geometry of 5.4, an 8th-degree polynomial solve, and then `kepler_U` + the Lagrange coefficients (Chapter 3) to refine the answer.

## Read first

| Symbol | Meaning |
|---|---|
| $\hat{\boldsymbol{\rho}}_1,\hat{\boldsymbol{\rho}}_2,\hat{\boldsymbol{\rho}}_3$ | unit line-of-sight directions at $t_1,t_2,t_3$ (known) |
| $\mathbf{R}_1,\mathbf{R}_2,\mathbf{R}_3$ | observer (site) positions at the three times |
| $\rho_1,\rho_2,\rho_3$ | the slant ranges — the **unknowns** |
| $\tau_1=t_1-t_2,\ \tau_3=t_3-t_2,\ \tau=\tau_3-\tau_1$ | time intervals |
| $D, D_0, A, B$ | bookkeeping scalars/matrix built from the inputs |
| output | $(\mathbf{r},\mathbf{v})$ improved, and $(\mathbf{r}_{\text{old}},\mathbf{v}_{\text{old}})$ before refinement |

**The relation tying it together:** $\;\mathbf{r}_i=\mathbf{R}_i+\rho_i\,\hat{\boldsymbol{\rho}}_i$ — same as 5.4, but now the $\rho_i$ are unknown and the dynamics must supply them.

## The picture (directions known, distances unknown)

A telescope gives **angles, not range**: at each time you know *which way* the object is ($\hat{\boldsymbol\rho}_i$) but not *how far* ($\rho_i$). Three sight-lines over a short arc, plus the fact that the object obeys Kepler's laws, is just enough to pin the distances down.

Gauss's two-stage strategy:

- **Algorithm 5.5 (initial estimate).** Approximate the Lagrange $f,g$ by their leading time-series (so they depend only on the middle distance $r_2$). Eliminating $\rho_1,\rho_3$ leaves $\rho_2$ as a function of $r_2$; substituting into $r_2=|\mathbf{R}_2+\rho_2\hat{\boldsymbol\rho}_2|$ collapses everything to a single **8th-degree polynomial in $r_2$**. Its positive real root gives the ranges, hence a first state.
- **Algorithm 5.6 (iterative improvement).** Replace the truncated $f,g$ with the *exact* universal-variable ones (`kepler_U` + Lagrange), recompute the ranges, and repeat until they stop changing.

The figure shows the idea: solid arrows are the known directions; the dashed segments are the unknown ranges the method solves for.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "..")
# Prerequisites you already built (used by the iterative-improvement stage):
from curtis.algorithms.alg_3_3_kepler_U import kepler_U
from curtis.algorithms.lagrange_coefficients import f_and_g
from curtis.algorithms.alg_4_1_coe_from_sv import coe_from_sv

def posroot(Roots):
    """Return the positive real root from numpy.roots output."""
    pr = [r.real for r in Roots if abs(r.imag) < 1e-9 and r.real > 0]
    if not pr:
        raise ValueError("no positive real root")
    if len(pr) > 1:
        print("multiple positive roots, using the first:", pr)
    return pr[0]

In [ ]:
# Angles-only schematic: known directions (solid), unknown ranges (dashed).
O = np.array([0.0, 0.0])
P = np.array([[2.0, 1.15], [2.45, 1.95], [2.7, 2.85]])     # object at t1, t2, t3
t = np.array([0, 1, 2]); ts = np.linspace(-0.25, 2.25, 100)
ox = np.polyval(np.polyfit(t, P[:,0], 2), ts)
oy = np.polyval(np.polyfit(t, P[:,1], 2), ts)

fig, ax = plt.subplots(figsize=(6.8, 6.0))
ax.plot(ox, oy, color='#73726c', lw=1.6)                   # orbit arc
ax.plot(0, 0, 'o', color='#3B8BD4', ms=9)
labels = ['t1', 't2', 't3']; cols = ['#888780', '#1D9E75', '#888780']
for Pi, lbl, col in zip(P, labels, cols):
    u = Pi/np.linalg.norm(Pi)
    ax.annotate('', xy=tuple(1.0*u), xytext=(0,0),
                arrowprops=dict(arrowstyle='-|>', color='#378ADD', lw=2))   # known direction
    ax.plot([1.0*u[0], Pi[0]], [1.0*u[1], Pi[1]], color='#D85A30', lw=1.4, ls='--')  # unknown range
    ax.plot(*Pi, 'o', color=col, ms=6)
    ax.annotate(lbl, Pi, textcoords='offset points', xytext=(7,4), color=col)
ax.annotate('observer', (0,0), textcoords='offset points', xytext=(6,-13), color='#3B8BD4')
ax.annotate(r'known direction $\hat\rho$', 1.0*P[2]/np.linalg.norm(P[2]) + np.array([-0.1,-0.35]),
            color='#378ADD', fontsize=10)
ax.annotate(r'unknown range $\rho$', (1.9, 2.35), color='#D85A30', fontsize=10)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Gauss: three sight-lines, ranges unknown -> the orbit')
plt.show()

## See it — the dynamics supplies the missing distances

Each blue arrow is a measured direction $\hat{\boldsymbol\rho}_i$; you know exactly *where to look* at each time but not *how far*. The dashed orange segments — the ranges $\rho_i$ — are what you do not measure. Gauss's method uses the requirement that the three resulting positions $\mathbf{r}_i=\mathbf{R}_i+\rho_i\hat{\boldsymbol\rho}_i$ lie on **one Keplerian orbit** to solve for those dashed lengths, turning pure angles into a full state vector.

## Where these equations come from

### Set-up: cross products and the $D$ matrix
The sight-line cross products $\mathbf{p}_1=\hat{\boldsymbol\rho}_2\times\hat{\boldsymbol\rho}_3$ etc. and $D_0=\hat{\boldsymbol\rho}_1\cdot\mathbf{p}_1$ measure how independent the three directions are (if $D_0\approx0$ the geometry is degenerate). The matrix $D_{ij}=\mathbf{R}_i\cdot\mathbf{p}_j$ packages the projections of the observer positions onto those directions.

### Stage 1 — the 8th-degree polynomial (Algorithm 5.5)
Approximating $f_{1,3}\approx1-\tfrac12\tfrac{\mu}{r_2^3}\tau_{1,3}^2$ and $g_{1,3}\approx\tau_{1,3}-\tfrac16\tfrac{\mu}{r_2^3}\tau_{1,3}^3$ makes the Lagrange coefficients depend only on the unknown $r_2$. Eliminating $\rho_1,\rho_3$ gives $\rho_2=A+\mu B/r_2^3$, and substituting into $r_2^2=(\mathbf{R}_2+\rho_2\hat{\boldsymbol\rho}_2)\cdot(\dots)$ yields
$$r_2^8+a\,r_2^6+b\,r_2^3+c=0\qquad(5.116),$$
with $a=-(A^2+2AE+R_2^2)$, $b=-2\mu B(A+E)$, $c=-(\mu B)^2$ (Eq 5.117). Its **positive real root** is the middle distance; back-substitution gives all three ranges and a first $\mathbf{v}_2$ (Eq 5.118).

### Stage 2 — iterative improvement (Algorithm 5.6)
The truncated series are only first guesses. Now compute the *exact* Lagrange coefficients from the current state using `kepler_U` (universal anomaly) and `f_and_g`, average them with the previous values for stability, recompute the ranges via the coefficients $c_1,c_3$ (Eqs 5.96–5.97, 5.109a–5.111a), and loop until the ranges stop moving.

### A book erratum to know
The printed code for Eq 5.114 (the $\rho_3$ formula) has the denominator $6x^3+\mu(\tau^2-\tau_1^2)$, but the §5.10 derivation and the book's own printed output use $(\tau^2-\tau_3^2)$. The skeleton below uses the correct $(\tau^2-\tau_3^2)$ — without it the un-improved $\mathbf{v}_{\text{old}}$ comes out wrong (the converged answer is unaffected because Stage 2 recomputes the ranges).

## Step by step (in code order)

**Algorithm 5.5 — initial estimate:** time intervals $\tau_1,\tau_3,\tau$ → cross products $\mathbf{p}_i$ → $D_0$ → $D$ matrix → $E$ → $A,B$ → polynomial coeffs $a,b,c$ → `np.roots` + `posroot` ($=r_2$) → truncated $f_{1,3},g_{1,3}$ → ranges $\rho_2,\rho_1,\rho_3$ → positions $\mathbf{r}_i$ → $\mathbf{v}_2$ (save these as `r_old, v_old`).

**Algorithm 5.6 — iterate:** while the ranges keep changing — recompute $\alpha,\chi$ via `kepler_U`, exact $f,g$ via `f_and_g`, average with previous, get $c_1,c_3$, update $\rho_i$, update $\mathbf{r}_i,\mathbf{v}_2$.

The detailed formulas are in the comments of the skeleton below.

**↓ Now type the implementation below.** This is the longest one — work through Stage 1, check `r_old`/`v_old` against the "without improvement" numbers, then add Stage 2. `posroot`, `kepler_U`, `f_and_g` are provided above. Answer key linked; peek only after you try.

In [ ]:
def gauss(Rho1, Rho2, Rho3, R1, R2, R3, t1, t2, t3, mu, tol=1.0e-8, nmax=1000):
    """Angles-only OD. Returns (r, v, r_old, v_old): improved and un-improved state at t2."""
    Rho1, Rho2, Rho3 = map(lambda a: np.asarray(a, float), (Rho1, Rho2, Rho3))
    R1, R2, R3 = map(lambda a: np.asarray(a, float), (R1, R2, R3))

    # ===== Algorithm 5.5 — initial estimate =====
    # 1. Time intervals (Eqs 5.98, 5.101):
    #        tau1 = t1 - t2;  tau3 = t3 - t2;  tau = tau3 - tau1
    # 2. Sight-line cross products:
    #        p1 = cross(Rho2, Rho3);  p2 = cross(Rho1, Rho3);  p3 = cross(Rho1, Rho2)
    # 3. Do = dot(Rho1, p1)                                              (Eq 5.108)
    # 4. D[i][j] = dot(R_i, p_j)   ->  3x3 array                         (Eqs 5.109b-5.111b)
    # 5. E = dot(R2, Rho2)                                               (Eq 5.115b)
    # 6. A = (1/Do)*(-D[0,1]*tau3/tau + D[1,1] + D[2,1]*tau1/tau)        (Eq 5.112b)
    #    B = (1/(6*Do))*(D[0,1]*(tau3**2 - tau**2)*tau3/tau
    #                    + D[2,1]*(tau**2 - tau1**2)*tau1/tau)           (Eq 5.112c)
    # 7. Polynomial coefficients (Eq 5.117):
    #        a = -(A**2 + 2*A*E + np.linalg.norm(R2)**2)
    #        b = -2*mu*B*(A + E)
    #        c = -(mu*B)**2
    # 8. Eighth-degree polynomial (Eq 5.116), take the positive real root:
    #        Roots = np.roots([1, 0, a, 0, 0, b, 0, 0, c]);  x = posroot(Roots)   # x = r2
    # 9. Truncated Lagrange series (Eqs 5.99, 5.100):
    #        f1 = 1 - 0.5*mu*tau1**2/x**3;  f3 = 1 - 0.5*mu*tau3**2/x**3
    #        g1 = tau1 - mu*(tau1/x)**3/6;  g3 = tau3 - mu*(tau3/x)**3/6
    # 10. Slant ranges:
    #        rho2 = A + mu*B/x**3                                        (Eq 5.112a)
    #        rho1 = (1/Do)*((6*(D[2,0]*tau1/tau3 + D[1,0]*tau/tau3)*x**3
    #                 + mu*D[2,0]*(tau**2 - tau1**2)*tau1/tau3)
    #                 / (6*x**3 + mu*(tau**2 - tau3**2)) - D[0,0])        (Eq 5.113)
    #        rho3 = (1/Do)*((6*(D[0,2]*tau3/tau1 - D[1,2]*tau/tau1)*x**3
    #                 + mu*D[0,2]*(tau**2 - tau3**2)*tau3/tau1)
    #                 / (6*x**3 + mu*(tau**2 - tau3**2)) - D[2,2])        (Eq 5.114; tau3, not tau1!)
    # 11. Positions:  r1 = R1 + rho1*Rho1;  r2 = R2 + rho2*Rho2;  r3 = R3 + rho3*Rho3   (Eq 5.86)
    # 12. v2 = (-f3*r1 + f1*r3)/(f1*g3 - f3*g1)                          (Eq 5.118)
    #     r_old = r2.copy();  v_old = v2.copy()

    # ===== Algorithm 5.6 — iterative improvement =====
    # rho1_old, rho2_old, rho3_old = rho1, rho2, rho3
    # diff1 = diff2 = diff3 = 1.0;  n = 0
    # while (diff1 > tol and diff2 > tol and diff3 > tol) and n < nmax:
    #     n += 1
    #     ro = norm(r2); vo = norm(v2); vro = dot(v2, r2)/ro; alpha = 2/ro - vo**2/mu
    #     x1 = kepler_U(tau1, ro, vro, alpha, mu);  x3 = kepler_U(tau3, ro, vro, alpha, mu)
    #     ff1, gg1 = f_and_g(x1, tau1, ro, alpha, mu)
    #     ff3, gg3 = f_and_g(x3, tau3, ro, alpha, mu)
    #     f1 = (f1+ff1)/2; f3 = (f3+ff3)/2; g1 = (g1+gg1)/2; g3 = (g3+gg3)/2     # average old & new
    #     c1 = g3/(f1*g3 - f3*g1);  c3 = -g1/(f1*g3 - f3*g1)                     (Eqs 5.96, 5.97)
    #     rho1 = (1/Do)*(-D[0,0] + (1/c1)*D[1,0] - (c3/c1)*D[2,0])               (Eqs 5.109a-5.111a)
    #     rho2 = (1/Do)*(-c1*D[0,1] + D[1,1] - c3*D[2,1])
    #     rho3 = (1/Do)*(-(c1/c3)*D[0,2] + (1/c3)*D[1,2] - D[2,2])
    #     r1 = R1 + rho1*Rho1; r2 = R2 + rho2*Rho2; r3 = R3 + rho3*Rho3
    #     v2 = (-f3*r1 + f1*r3)/(f1*g3 - f3*g1)
    #     diff1 = abs(rho1 - rho1_old); diff2 = abs(rho2 - rho2_old); diff3 = abs(rho3 - rho3_old)
    #     rho1_old, rho2_old, rho3_old = rho1, rho2, rho3

    # return r2, v2, r_old, v_old
    raise NotImplementedError("type Stage 1, then Stage 2, then delete this line")

## Verify — Example 5.11

The inputs are built from three optical observations (right ascension, declination, and the site's local sidereal time at each instant) — that setup is given below. Then your `gauss` is called.

**Expected — without iterative improvement:** $\mathbf{r}=[5659.03,\,6533.74,\,3270.15]$ km, $\mathbf{v}=[-3.90774,\,5.05735,\,-2.22224]$ km/s.

**Expected — with iterative improvement:** $\mathbf{r}=[5662.04,\,6537.95,\,3269.05]$ km, $\mathbf{v}=[-3.88542,\,5.12141,\,-2.2434]$ km/s; orbit $a=9999.5$ km, $e=0.0999909$, $i=30.0°$.

In [ ]:
# --- Provided: build the observation vectors for Example 5.11 ---
deg = np.pi/180
mu  = 398600.0
Re  = 6378.0
f   = 1/298.26
H   = 1.0
phi = 40.0*deg
t   = np.array([0.0, 118.104, 237.577])
ra  = np.array([43.5365, 54.4196, 64.3178])*deg
dec = np.array([-8.78334, -12.0739, -15.1054])*deg
theta = np.array([44.5065, 45.000, 45.4992])*deg

fac1 = Re/np.sqrt(1 - (2*f - f*f)*np.sin(phi)**2)
fac2 = Re*(1-f)**2/np.sqrt(1 - (2*f - f*f)*np.sin(phi)**2)
R = np.zeros((3,3)); Rho = np.zeros((3,3))
for i in range(3):
    R[i] = [(fac1+H)*np.cos(phi)*np.cos(theta[i]),
            (fac1+H)*np.cos(phi)*np.sin(theta[i]),
            (fac2+H)*np.sin(phi)]
    Rho[i] = [np.cos(dec[i])*np.cos(ra[i]), np.cos(dec[i])*np.sin(ra[i]), np.sin(dec[i])]

# --- Your gauss ---
r, v, r_old, v_old = gauss(Rho[0], Rho[1], Rho[2], R[0], R[1], R[2], t[0], t[1], t[2], mu)

print("without improvement:")
print(f"  r = [{r_old[0]:.6g} {r_old[1]:.6g} {r_old[2]:.6g}]   (expect [5659.03  6533.74  3270.15])")
print(f"  v = [{v_old[0]:.6g} {v_old[1]:.6g} {v_old[2]:.6g}]   (expect [-3.90774  5.05735  -2.22224])")
print("with improvement:")
print(f"  r = [{r[0]:.6g} {r[1]:.6g} {r[2]:.6g}]   (expect [5662.04  6537.95  3269.05])")
print(f"  v = [{v[0]:.6g} {v[1]:.6g} {v[2]:.6g}]   (expect [-3.88542  5.12141  -2.2434])")

assert np.allclose(r_old, [5659.03, 6533.74, 3270.15], atol=1e-1)
assert np.allclose(v_old, [-3.90774, 5.05735, -2.22224], atol=1e-3)
assert np.allclose(r, [5662.04, 6537.95, 3269.05], atol=1e-1)
assert np.allclose(v, [-3.88542, 5.12141, -2.2434], atol=1e-3)

coe = coe_from_sv(r, v, mu)
print(f"\norbit: a={coe[6]:.6g} km  e={coe[1]:.6g}  i={coe[3]/deg:.5g} deg")
assert abs(coe[6] - 9999.5) < 1.0 and abs(coe[1] - 0.0999909) < 1e-4
print("\nAll checks passed ✔")

## What this confirms

- **Angles alone determine an orbit.** Three line-of-sight directions plus Kepler's laws are enough to recover the missing ranges and the full state — no range measurement required.
- The method is a **two-stage solve**: an algebraic first guess (the 8th-degree polynomial), then a dynamical refinement (`kepler_U` + Lagrange) — and you watched the second stage move the answer onto the true orbit.
- It is the synthesis of the whole book so far: line-of-sight geometry (5.4), root-finding, and universal-variable propagation (Chapter 3).

**That closes Chapter 5.** The notebook series now spans the two-body core (Ch 3), state ⇄ elements (Ch 4), and preliminary orbit determination (Ch 5). The remaining Appendix-D algorithms live in **Chapter 8** (interplanetary): planetary ephemeris (8.1) and the patched-conic trajectory from planet to planet (8.2).